In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# data visualizations
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf
import ydf
from sklearn.model_selection import train_test_split

I0000 00:00:1793075786.866113   59344 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE3 SSE4.1 SSE4.2 AVX AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


ModuleNotFoundError: No module named 'ydf'

In [ ]:
import os
for dirname, _, filenames in os.walk('data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

data/sample_submission.csv
data/test.csv
data/train.csv


In [ ]:
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')


In [ ]:
x = train.copy()
x['TotalSpent'] = x['RoomService'] + x['FoodCourt'] + x['ShoppingMall'] + x['Spa'] + x['VRDeck']

In [ ]:
mask = (x['TotalSpent'] == 0) & (x['VIP'] == False)
print(f"num of ppl who spend 0 and are NOT VIP: {x[mask].shape}")
mask = (x['TotalSpent'] == 0) & (x['VIP'] == True)
print(f"num of ppl who spend 0 and are VIP: {x[mask].shape}")

num of ppl who spend 0 and are NOT VIP: (3137, 15)
num of ppl who spend 0 and are VIP: (24, 15)


In [ ]:
y = train.iloc[7]

In [ ]:
def preprocess(curr_df):
    df = curr_df.copy()
    # cut unneccessary features
    df = df.drop(['PassengerId', 'Name'], axis=1)
    # 0 or 1 for boolean values
    try:
        df["Transported"] = df["Transported"].astype(int)
    except Exception as e:
        print(f"Transported column not found, this is fine for test dataset: {e}")
    # fill na values for numerical values
    df[["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]] = df[["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]].fillna(0)
    
    # find if there's a better way to deduce VIP status
    df["VIP"] = df["VIP"].fillna(0)
    df["VIP"] = df["VIP"].astype(int)

    # use smart deduction to fill CryoSleep values
    df['TotalSpent'] = df['RoomService'] + df['FoodCourt'] + df['ShoppingMall'] + df['Spa'] + df['VRDeck']
    df['CryoSleep'] = df['CryoSleep'].fillna(df['TotalSpent'] == 0)

    # split Cabin into 3 features
    df[["Deck", "Cabin_num", "Side"]] = df['Cabin'].str.split('/', expand=True)
    df = df.drop('Cabin', axis=1)
    
    # Fill missing values in categorical columns with a string placeholder
    categorical_cols = ['HomePlanet', 'Destination', 'Deck', 'Cabin_num', 'Side']
    for col in categorical_cols:
        df[col] = df[col].fillna("Unknown").astype(str)
        
    return df

In [ ]:
processed_train = preprocess(train)

In [ ]:
train_split, test_split = train_test_split(
    processed_train, 
    test_size=0.2, 
    random_state=42,
    stratify=processed_train['Transported'])

In [ ]:
learner = ydf.GradientBoostedTreesLearner(label = "Transported")

In [ ]:
model = learner.train(train_split)

Train model on 6954 examples
Model trained in 0:00:00.514773


In [ ]:
model.describe()

In [ ]:
model.evaluate(test_split)

Label \ Pred,0,1
0,689,174
1,153,723


In [ ]:
processed_test = preprocess(test)

Transported column not found, this is fine for test dataset: 'Transported'


In [ ]:
predictions = model.predict(processed_test)

In [ ]:
n_predictions = (predictions > .5).astype(bool)

In [ ]:
pd.DataFrame({'PassengerId': test.PassengerId, 'Transported': n_predictions})

,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True
...,...,...
4272,9266_02,True
4273,9269_01,False
4274,9271_01,True
4275,9273_01,True


In [ ]:
pd.DataFrame({'PassengerId': test.PassengerId, 'Transported': n_predictions}).to_csv('submission.csv', index=False)